In [ ]:
import scipy as sp
import numpy as np
import thewalrus as wr
import itertools
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import ipywidgets as widgets
from datetime import datetime
%matplotlib widget

In [ ]:
def T(n):
    """Defines conversion matrix for thewalrus
    T.T@sigma@T is our converted matrix
    T@sigma@T.T converts back
    """
    v1 = np.array([[1,0]])
    v2 = np.array([[0,1]])
    T1 = sp.linalg.block_diag(*([v1]*n))
    T2 = sp.linalg.block_diag(*([v2]*n))
    T = np.block([[T1],[T2]])
    return T
def sigma(η,ns,nb):
    return np.array([
        [1 +2*η*ns + 2*nb,0,-2*np.sqrt(ns*(1+ns)*η),0],
        [0,1 +2*η*ns + 2*nb,0,2*np.sqrt(ns*(1+ns)*η)],
        [-2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns,0],
        [0,2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns]
    ])


def sigmasu(η,ns,nb):
    bkrd = 1+2*ns*((1+ns)*(1+np.sqrt(η))**2+nb)
    corr = 2*np.sqrt(ns*(1+ns))*(1+np.sqrt(η)+nb+ns*(1+np.sqrt(η))**2)
    return np.array([
        [bkrd + 2*nb,0,-corr,0],
        [0,bkrd+ 2*nb,0,corr],
        [-corr,0,bkrd+2*ns*(1-η),0],
        [0,corr,0,bkrd+2*ns*(1-η)]
    ])

def vec_Ps_generic_jac_gen(cov,ns,nb,maxval):
    means = np.zeros(4)
    T2 = T(2)
    def vec_Ps_su_jac(x):
        xvals = x[0]
        output = np.zeros((maxval,maxval,*xvals.shape))
        for i,val in np.ndenumerate(xvals):
            covval = T2@cov(val,ns,nb)@T2
            Ps = wr.quantum.probabilities(means,covval,maxval)
            output[:,:,*i] = Ps
        return output
    return vec_Ps_su_jac
def FI_generic(etas,nss,nbs,cov,maxval=10):
    indices = np.arange(maxval)
    etagrid,nsgrid,nbgrid = np.meshgrid(etas,nss,nbs,indexing='ij')
    ds = np.zeros((len(nss),len(nbs),maxval,maxval,len(etas)))
    for (i,ns),(j,nb) in tqdm(itertools.product(enumerate(nss),enumerate(nbs)),total=len(nss)*len(nbs),smoothing=.01):
        vec_ps_su_jac = vec_Ps_generic_jac_gen(cov,ns,nb,maxval)
        derivres = sp.differentiate.jacobian(vec_ps_su_jac,etas,initial_step=1e-9)
        ds[i,j,:,:,:] = derivres.df
    T2 = T(2)
    means = np.zeros(4)
    ps = np.zeros((len(nss),len(nbs),maxval,maxval,len(etas)))
    for (i,ns),(j,nb),(k,eta) in tqdm(itertools.product(enumerate(nss),enumerate(nbs),enumerate(etas)),total=len(nss)*len(nbs)*len(etas),smoothing=.01):
        covval = T2@cov(eta,ns,nb)@T2
        Ps = wr.quantum.probabilities(means,covval,maxval)
        ps[i,j,:,:,k] = Ps
    presum = ds**2/ps
    presum[~np.isfinite(presum)] = 0
    fi = np.sum(presum,axis=(2,3))
    return fi


In [ ]:
Netah = 800
Neta = 2*Netah
Ns = 4
Nb = 4
etahalf = np.logspace(-7,-.301,Netah)
etavals = np.append(etahalf,np.flip(1-etahalf))
nsvals = np.logspace(-5,-1,Ns,dtype=np.double)
nbvals = np.insert(np.logspace(-4,0,Nb-1,dtype=np.double),0,0)
maxval = 9

In [ ]:
FIspn = FI_generic(etavals,nsvals,nbvals,sigma,maxval)
FIssu = FI_generic(etavals,nsvals,nbvals,sigmasu,maxval)

In [ ]:
nsslider = widgets.IntSlider(
    value=0,
    min=0,
    max=Ns-1,
    step=1,
    description='Ns index',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
nbslider = widgets.IntSlider(
    value=0,
    min=0,
    max=Nb-1,
    step=1,
    description='Nb index:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

def display_graph(nsi,nbi):
    fig = plt.figure()
    plt.xlabel("Loss Parameter (η)")
    plt.ylabel("FI Difference")
    plt.semilogy(etavals,FIssu[nsi,nbi,:]-FIspn[nsi,nbi,:])
    fig.title = f"FI Difference for ns {nsvals[nsi]} and nb {nbvals[nbi]}"
    return fig


In [ ]:
widgets.interact(display_graph,nsi = nsslider,nbi = nbslider,continuous=False)

In [ ]:
# Save data
filename = "../_data/su11vspnr" + datetime.now().strftime("%Y_%m_%d-%H_%M")# Append the current date and time to avoid filename conflicts
np.savez_compressed(filename,etavals=etavals,nsvals = nsvals,nbvals = nbvals, FIspn = FIspn,FIssum = FIssu)